## Colab setup

Auto-detects whether this is running in Colab. If so, clones the repo and reconstructs `.env` from a Colab secret. If running locally, this is a no-op (assumes the repo is already checked out and `.env` already exists).


In [1]:
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    repo_dir = "/content/change-my-view-2025"
    work_dir = repo_dir + "/this work"
    if not os.path.exists(repo_dir):  # skip re-cloning if this cell runs twice in the same session
        !git clone -b final-project https://github.com/jct-nlp/change-my-view-2025.git {repo_dir}
    if os.getcwd() != work_dir:  # skip re-cd-ing if already there
        %cd $work_dir
    with open("../.env", "w") as f:
        f.write(f"GEMINI_API_KEY={userdata.get('GEMINI_API_KEY')}")
else:
    print("Not running in Colab — skipping repo clone and .env setup (assuming local checkout already has both).")

Not running in Colab — skipping repo clone and .env setup (assuming local checkout already has both).


# 03.2 — Feature Engineering (Heavy / GPU)

This notebook continues from `03.1_feature_engineering_light.ipynb`, which computes all the CPU-only features and saves intermediate files. This notebook adds the GPU-heavy features — RoBERTa and BART/XLM-RoBERTa classification, and BERT document embeddings — merges `op_df` into `comments_df`, and produces the final `cmv_comments_df.csv`.

**Run `03.1` first**, and set the runtime type to GPU (Runtime → Change runtime type → GPU) before running this notebook.


In [ ]:
import pandas as pd, json

comments_df = pd.read_csv('../results/comments_df_light.csv')
op_df = pd.read_csv('../results/op_df_light.csv')

with open('../results/untrusted_features_light.json') as f:
    untrusted_features = json.load(f)

separator_token = "[SEP]"  # marks the boundary between thread text and final comment when concatenated

print(f'Loaded comments_df_light.csv ({len(comments_df)} rows), op_df_light.csv ({len(op_df)} rows)')
print(f'untrusted_features so far: {untrusted_features}')


## Shared utilities

Defined in `03.1` too, since its own light sections need it; needed again here for the RoBERTa tone section below.

In [ ]:
def get_comment_and_thread(comment_first=False, is_convincing=None, include_sep=True):
  if is_convincing is not None:
    df = comments_df[comments_df["is_convincing"] == is_convincing]
  else:
    df = comments_df

  if include_sep:
    sep = "\n" + separator_token + "\n"
  else:
    sep = "\n"

  if comment_first:
    return df['final_comment'] + sep + df['thread_text'].fillna("")

  return df['thread_text'].fillna("") + sep + df['final_comment']

## 6.2 Tone
We would like to create a feature for the tone of the text. The tone can be, for example, logical, passionate, humorous etc.
We will try a pre-trained model based on RoBERTa.

(A Gemini-based follow-up to this attempt continues in `03.1_feature_engineering_light.ipynb`, since it only needs API calls, not GPU.)

### RoBERTa based model

In [ ]:
!pip install torch transformers

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")

def get_tone(text):
    return classifier(text, truncation=True, padding=True)  # since our texts may be long and Roberta is limited to 512 tokens, we truncate any excessive text

text = get_comment_and_thread(comment_first=True) # the text may be truncated so we put the convincing comment first - it's more important
comments_df["tone"] = text.apply(get_tone)


In [ ]:
print(comments_df['tone'])


In [ ]:
print(comments_df['tone'][0])

In [ ]:
comments_df['tone_label'] = comments_df['tone'].apply(lambda x: x[0]['label'] if isinstance(x, list) and len(x) > 0 else None)

In [ ]:
comments_df['tone_label']

In [ ]:
import matplotlib.pyplot as plt

def plot_category_histogram(df, column_name, display_name):
    """
    Plots a histogram showing the distribution of convincing and non-convincing comments for each category in the specified column.

    Args:
        df: The pandas DataFrame containing the data.
        column_name: The name of the column containing the categories.
    """

    convincing_counts = df[df['is_convincing'] == 1][column_name].value_counts()
    non_convincing_counts = df[df['is_convincing'] == 0][column_name].value_counts()

    categories = convincing_counts.index.union(non_convincing_counts.index)
    convincing_counts = convincing_counts.reindex(categories, fill_value=0)
    non_convincing_counts = non_convincing_counts.reindex(categories, fill_value=0)

    width = 0.4
    x = range(len(categories))
    fig, ax = plt.subplots(figsize=(12,6))
    rects1 = ax.bar([i - width/2 for i in x], convincing_counts, width, label='Convincing', color='green')
    rects2 = ax.bar([i + width/2 for i in x], non_convincing_counts, width, label='Non-Convincing', color='red')

    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate('{}'.format(height),
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha='center', va='bottom')
    autolabel(rects1)
    autolabel(rects2)

    ax.set_xlabel(display_name)
    ax.set_ylabel('Number of Comments')
    ax.set_title(f'Number of Convincing and Non-Convincing Comments per {display_name}')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_category_histogram(comments_df, 'tone_label', 'Tone')


In [ ]:
def show_tone_examples(tone_label):
  for index, row in comments_df[comments_df['tone'].apply(lambda x: isinstance(x, list) and len(x) > 0 and x[0]['label'] == tone_label and x[0]['score'] > 0.85)].iterrows():
    print(f"Example for {tone_label}:")
    print(row['final_comment'])
    print("-" * 20)
    break

In [ ]:
show_tone_examples("fear")
show_tone_examples("joy")
show_tone_examples("surprise")


We are not very satisfied with the results of classification, it seems that the classification is not very accurate and we want to try a different approach, using Google AI Studio.

In [ ]:
untrusted_features.append('tone_label')


## 6.3 Style - formal / informal
We would like to create a feature for the style of the text, in other words is it formal or informal text.

For this we will use the model facebook/bart-large-mnli that has been trained on a wide range of general text categories.

We will use this model with zero-shot classification, by defining labels "formal" and "informal" and classify the texts into these categories.

In [ ]:
import re
def do_calc_feature(feature, func, remove_urls=False, switch_order=False):
  for index, row in comments_df.iterrows():
    if pd.isna(row[feature]):  # calculate only if wasn't calculated yet
      print(f"Calculating {feature} for index {index}")
      if not isinstance(row['thread_text'], str):
        text = row['final_comment']
      elif switch_order:
        text = row['final_comment'] + "\n" + row['thread_text']
      else:
        text = row['thread_text'] + "\n" + row['final_comment']

      if remove_urls:
        text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
      comments_df.at[index, feature] = func(text)

In [ ]:
from transformers import pipeline

# Load zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = ["formal", "informal"]

def get_style(text):
    return classifier(text, candidate_labels)


In [ ]:
comments_df['style'] = None

In [ ]:
do_calc_feature("style", get_style)

In [ ]:
comments_df['style'].iloc[0]

In [ ]:
import json
import ast

def convert_string_to_json(string):
  data = ast.literal_eval(string)
  data_obj = json.dumps(data, indent=4)
  return json.loads(data_obj)

def get_style_label(style):
  if isinstance(style, str):
    style = convert_string_to_json(style)
  return style['labels'][style['scores'].index(max(style['scores']))]

In [ ]:
get_style_label(comments_df['style'].iloc[0])

In [ ]:
comments_df['style_label'] = comments_df['style'].apply(get_style_label)

In [ ]:
plot_category_histogram(comments_df, 'style_label', 'Style')

We see that most of the comments are informal, which makes sense for this type of communication.

In [ ]:
def show_style_example(style):
  print(f"Example for {style}:")
  for _, row in comments_df[comments_df['style_label'] == style].iterrows():
    style = convert_string_to_json(row['style'])
    if max(style['scores']) > 0.95:
      print(row['final_comment'])
      print("-" * 20)
      break

show_style_example("formal")
show_style_example("informal")

## 6.8 Use of Persuasive Language

We want to identify if the comment and context include use of persuasive language. To do this, we will use a HuggingFace model: https://huggingface.co/chreh/persuasive_language_detector.

From the model card:
> Given a sentence, our model predicts whether or not the sentence contains "persuasive" language, or language designed to elicit emotions or change readers' opinions. The model was tuned on the SemEval 2020 Task 11 dataset. However, we preprocessed the dataset to adapt it from multilabel technique classification and span-classification to our binary classification task.

The model has two flavours, BERT and XLM-RoBERTa. Based on the documentation, the latter performs better and faster, so we will use it.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained("chreh/persuasive_language_detector", revision="roberta")


In [ ]:
import torch

max_length = 512
def classify_text(text):
    # The model gets only up to 512 tokens
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class


In [ ]:
 text = comments_df.iloc[0]['final_comment']
 classify_text(text)

In [ ]:
text

Let's apply the featute on all rows:

In [ ]:
comments_df['use_of_persuasive_lang'] = None

In [ ]:
do_calc_feature("use_of_persuasive_lang", classify_text, switch_order=True)

In [ ]:
comments_df['use_of_persuasive_lang']

In [ ]:
plot_category_histogram(comments_df, 'use_of_persuasive_lang', ' Use of Persuasive Language')

In [ ]:
# convert comments_df['use_of_persuasive_lang'] to ints
comments_df['use_of_persuasive_lang'] = comments_df['use_of_persuasive_lang'].apply(lambda x: 1 if x == '1' else 0)

## 6.9 Document Embedding

We want that our model will also get some kind of representation of the original text, not only features extracted from it. Therefore we will include the docuemnt embedding in the features list.

First we will calculate the embedding using a variation of the BERT model, bert-base-uncased, which is a well-studied, strong contextual embeddings, suitable for general English NLP tasks.

This model provides encoding of dimension 768.

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text, chunk_size=256, overlap=128):
    """
    Convert a long document into a single embedding using BERT with chunking.

    Parameters:
    - text (str): The long document
    - chunk_size (int): Max tokens per chunk (default: 256)
    - overlap (int): Overlapping tokens between chunks (default: 128)

    Returns:
    - Aggregated document embedding (numpy array of shape [768])
    """
    # Tokenize document without truncation
    tokens = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]

    # Split into chunks with overlap
    chunk_embeddings = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk = tokens[i : i + chunk_size]  # Get chunk

        encoded_input = {
            "input_ids": chunk.unsqueeze(0),  # Add batch dimension
            "attention_mask": torch.ones_like(chunk).unsqueeze(0)  # Mask for valid tokens
        }

        # Forward pass through BERT
        with torch.no_grad():
            outputs = model(**encoded_input)

        # Extract last hidden state
        token_embeddings = outputs.last_hidden_state  # Shape: [1, chunk_size, 768]
        chunk_embedding = token_embeddings.mean(dim=1)  # Mean pooling over tokens

        # Store chunk embedding
        chunk_embeddings.append(chunk_embedding.squeeze().numpy())

    # Aggregate all chunk embeddings (mean pooling over chunks)
    document_embedding = np.mean(chunk_embeddings, axis=0)

    return document_embedding

In [ ]:
# Example Usage
long_text = comments_df.iloc[0]['final_comment']
embedding = get_embedding(long_text)
print(embedding.shape)  # Expected output: (768,)

In [ ]:
comments_df['embedding'] = None

In [ ]:
do_calc_feature("embedding", get_embedding)

### Embedding Dimension Reduction with PCA
Since the size of the encoding is very big, and adding so many features to our models will not be practical, we will apply PCA on the embeddings.

To decide the PCA dimension, we will execute some statistical computation to understand what is the minimal dimension that will allow us to keep at least 85% of the information that the original embedding contains.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Extract all embeddings into a matrix
embedding_matrix = np.vstack(comments_df["embedding"].values)  # Shape: (num_samples, 768)

# Apply PCA
pca = PCA().fit(embedding_matrix)  # Fit PCA on the full embedding

# Compute cumulative explained variance
explained_variance = np.cumsum(pca.explained_variance_ratio_)

# Find the number of components that explain at least 85% variance
target_dim = np.argmax(explained_variance >= 0.85) + 1  # +1 because indexing starts at 0

# Plot the explained variance
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(explained_variance) + 1), explained_variance, marker="o")
plt.axhline(y=0.85, color="r", linestyle="--", label="85% variance threshold")
plt.axvline(x=target_dim, color="g", linestyle="--", label=f"Optimal Dim = {target_dim}")
plt.xlabel("Number of PCA Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.show()

print(f"Optimal number of components: {target_dim}")


Now we will apply the PCA on the embedding:

In [ ]:
pca = PCA(n_components=target_dim)
reduced_embeddings = pca.fit_transform(embedding_matrix)

# Convert back to DataFrame and merge
embedding_columns = [f"embedding_{i}" for i in range(target_dim)]
embedding_df = pd.DataFrame(reduced_embeddings, columns=embedding_columns)

# Merge with original dataset
# comments_df = comments_df.drop(columns=["embedding"]).reset_index(drop=True)
comments_df = pd.concat([comments_df, embedding_df], axis=1)

Original-post-level features 7.1, 7.2, and 7.4-7.7 were already computed in `03.1` (they're light). This is the one op-level feature that needs the GPU-based BART model from 6.3 above.

## 7.3 Style

In [ ]:
import re
def do_calc_feature_op(feature, func, remove_urls=False):
  for index, row in op_df.iterrows():
    if pd.isna(row[feature]):  # calculate only if wasn't calculated yet
      print(f"Calculating {feature} for index {index}")
      text = row['original_post']

      if remove_urls:
        text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
      op_df.at[index, feature] = func(text)

In [ ]:
op_df['style_op'] = None

In [ ]:
do_calc_feature_op("style_op", get_style)

In [ ]:
op_df['style_label_op'] = op_df['style_op'].apply(get_style_label)

## 7.8 Combine op_df into comments_df

In [ ]:
for index, row in comments_df.iterrows():
    # Find matching original_post in op_df
    matching_rows = op_df[op_df['original_post'] == row['original_post']]

    if matching_rows.empty:
      continue

    # Copy columns from op_df to comments_df
    for col in op_df.columns:
      if col != 'original_post':  # Skip the original_post column itself
        comments_df.loc[index, col] = matching_rows[col].iloc[0]


In [ ]:
comments_df.head()

Let's save the final dataframe.

In [ ]:
comments_df.to_csv('../results/cmv_comments_df.csv', index=False)
print('Saved cmv_comments_df.csv')


In [ ]:
import json
with open('../results/untrusted_features.json', 'w') as f:
    json.dump(untrusted_features, f, indent=2)
print(f'Saved untrusted_features.json: {untrusted_features}')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute correlation for numeric features only
comments_df_no_embedding = comments_df.loc[:, ~comments_df.columns.str.startswith('embedding_')]
corr_matrix = comments_df_no_embedding.select_dtypes(include=['number']).corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Heatmap (w/o embedding features)")
plt.show()